# Notebook 02: WOE/IV Feature Engineering
## LendingClub Credit Risk Analytics

This notebook implements **Weight of Evidence (WOE)** and **Information Value (IV)** analysis
for credit scorecard feature selection. This mirrors the feature engineering phase of
behavioral scorecard monitoring and loss forecasting in my prior role, where WOE/IV
was the standard methodology for VantageScore/FICO-style binning of utilization, DTI,
inquiries, and other credit bureau features.

### Key Principles:
- WOE transformation converts features to a common scale of log-odds
- IV quantifies each feature's predictive power for separating defaults from non-defaults
- Binning is fit **ONLY on training data** — then applied to val/test to prevent leakage
- `grade`, `sub_grade`, and `int_rate` are excluded from the scorecard (they are outputs
  of the credit decision, not independent predictors)

### Inputs:
- `data/processed/train.parquet` (training data only)
- `config.py` constants

### Outputs:
- `src/woe_binning.py` (reusable WOEBinner module)
- `data/processed/woe_binning_results.pkl`
- `data/processed/iv_summary.csv`
- `data/processed/train_woe.parquet`, `val_woe.parquet`, `test_woe.parquet`

## Setup: Imports and Configuration

In [1]:
import sys
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Project imports
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from config import *
from src.woe_binning import WOEBinner

print(f'Project root: {PROJECT_ROOT}')
print(f'Random state: {RANDOM_STATE}')

Project root: /Volumes/T7/Portfolio Projects/lendingclub-credit-risk
Random state: 42


## Step 1: Load Training Data

WOE binning must be fit **only on training data**. We load the time-based split
from Notebook 01 (train: 2007-2015, val: 2016, test: 2017-2018).

In [2]:
train = pd.read_parquet(DATA_PROCESSED_PATH / 'train.parquet')
val = pd.read_parquet(DATA_PROCESSED_PATH / 'val.parquet')
test = pd.read_parquet(DATA_PROCESSED_PATH / 'test.parquet')

y_train = train[TARGET_COL]
y_val = val[TARGET_COL]
y_test = test[TARGET_COL]

print(f'Train: {train.shape} — default rate: {y_train.mean():.4f}')
print(f'Val:   {val.shape} — default rate: {y_val.mean():.4f}')
print(f'Test:  {test.shape} — default rate: {y_test.mean():.4f}')

Train: (826606, 124) — default rate: 0.1843
Val:   (293105, 124) — default rate: 0.2329
Test:  (225639, 124) — default rate: 0.2129


## Step 2: Create Engineered Features

Before WOE binning, we create derived features that combine raw attributes
into more predictive ratios and flags.

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create derived features for WOE/IV analysis."""
    df = df.copy()

    # Credit history length in years
    if 'earliest_cr_line' in df.columns and 'issue_d' in df.columns:
        ecl = pd.to_datetime(df['earliest_cr_line'], errors='coerce')
        iss = pd.to_datetime(df['issue_d'], errors='coerce')
        df['credit_history_years'] = (iss - ecl).dt.days / 365.25

    # FICO average
    if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
        df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

    # Loan-to-income ratio
    if 'loan_amnt' in df.columns and 'annual_inc' in df.columns:
        df['loan_to_income'] = df['loan_amnt'] / df['annual_inc'].clip(lower=1)

    # Installment-to-income ratio (monthly)
    if 'installment' in df.columns and 'annual_inc' in df.columns:
        monthly_inc = df['annual_inc'].clip(lower=1) / 12
        df['installment_to_income'] = df['installment'] / monthly_inc

    # Total credit utilization
    if 'revol_bal' in df.columns and 'total_rev_hi_lim' in df.columns:
        df['total_credit_utilization'] = (
            df['revol_bal'] / df['total_rev_hi_lim'].clip(lower=1)
        )

    # Binary flags
    if 'delinq_2yrs' in df.columns:
        df['delinq_flag'] = (df['delinq_2yrs'] > 0).astype(int)
    if 'inq_last_6mths' in df.columns:
        df['recent_inquiry_flag'] = (df['inq_last_6mths'] > 2).astype(int)
    if 'dti' in df.columns:
        df['high_dti_flag'] = (df['dti'] > 30).astype(int)

    return df


train = engineer_features(train)
val = engineer_features(val)
test = engineer_features(test)

new_feats = ['credit_history_years', 'fico_avg', 'loan_to_income',
             'installment_to_income', 'total_credit_utilization',
             'delinq_flag', 'recent_inquiry_flag', 'high_dti_flag']
print('Engineered features:')
for f in new_feats:
    if f in train.columns:
        print(f'  {f}: mean={train[f].mean():.4f}, missing={train[f].isna().mean():.2%}')

Engineered features:
  credit_history_years: mean=16.2419, missing=0.00%
  fico_avg: mean=697.0178, missing=0.00%
  loan_to_income: mean=0.2465, missing=0.00%
  installment_to_income: mean=0.0920, missing=0.00%
  total_credit_utilization: mean=0.5551, missing=0.00%
  delinq_flag: mean=0.1905, missing=0.00%
  recent_inquiry_flag: mean=0.0607, missing=0.00%
  high_dti_flag: mean=0.0866, missing=0.00%


## Step 3: Define Candidate Features for WOE/IV

We include all PD model features from CLAUDE.md plus engineered features.
**Excluded**: leakage variables (post-origination outcomes) and non-feature columns.
**Also excluded from scorecard**: `grade`, `sub_grade`, `int_rate` — these are outputs
of the credit decision process, creating circular dependency.

In [4]:
# Leakage variables — NEVER use for PD model
LEAKAGE = {
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt',
    'last_pymnt_d', 'last_fico_range_high', 'last_fico_range_low',
    'next_pymnt_d', 'last_credit_pull_d', 'hardship_flag',
    'debt_settlement_flag',
}

# Non-feature columns
NON_FEATURES = {
    TARGET_COL, 'issue_d', 'issue_month', 'issue_year',
    'emp_title', 'title', 'zip_code', 'addr_state',
    'earliest_cr_line', 'application_type',
    'initial_list_status', 'disbursement_method',
}

# Scorecard exclusions (circular — assigned based on credit risk)
SCORECARD_EXCLUDE = {'grade', 'sub_grade', 'int_rate'}

# Get all numeric columns
all_numeric = train.select_dtypes(include=[np.number]).columns.tolist()

# Candidate features = numeric - leakage - non_features
candidates = [c for c in all_numeric
              if c not in LEAKAGE
              and c not in NON_FEATURES
              and c not in SCORECARD_EXCLUDE]

print(f'Total numeric columns: {len(all_numeric)}')
print(f'After removing leakage ({len(LEAKAGE)}): {len(all_numeric) - len(LEAKAGE & set(all_numeric))}')
print(f'After removing non-features: {len(candidates) + len(SCORECARD_EXCLUDE & set(all_numeric))}')
print(f'After removing scorecard exclusions: {len(candidates)}')
print(f'\nCandidate features ({len(candidates)}):')
for i, c in enumerate(sorted(candidates)):
    print(f'  {i+1:2d}. {c}')

Total numeric columns: 112
After removing leakage (17): 100
After removing non-features: 98
After removing scorecard exclusions: 97

Candidate features (97):
   1. A191RL1Q225SBEA
   2. CPIAUCSL
   3. CSUSHPINSA
   4. DFF
   5. UMCSENT
   6. UNRATE
   7. acc_now_delinq
   8. acc_open_past_24mths
   9. all_util
  10. annual_inc
  11. avg_cur_bal
  12. bc_open_to_buy
  13. bc_util
  14. chargeoff_within_12_mths
  15. collections_12_mths_ex_med
  16. credit_history_years
  17. delinq_2yrs
  18. delinq_amnt
  19. delinq_flag
  20. dti
  21. emp_length
  22. emp_length_unknown
  23. fico_avg
  24. fico_range_high
  25. fico_range_low
  26. funded_amnt
  27. funded_amnt_inv
  28. has_major_derog
  29. has_public_record
  30. has_recent_bc_delinq
  31. high_dti_flag
  32. il_util
  33. il_util_missing
  34. inq_fi
  35. inq_last_12m
  36. inq_last_6mths
  37. installment
  38. installment_features_missing
  39. installment_to_income
  40. loan_amnt
  41. loan_to_income
  42. max_bal_bc
  43. 

## Step 4: Apply WOE Binning to All Candidate Features

The WOEBinner uses decision tree-based optimal binning to find splits that
maximize separation between default and non-default groups. Monotonic bad-rate
ordering is enforced across bins — this is critical for scorecard development.

In [5]:
# Fit WOE binner on training data ONLY
X_train_candidates = train[candidates].copy()

binner = WOEBinner(max_bins=10, min_bin_pct=0.05, monotonic=True,
                   random_state=RANDOM_STATE)
binner.fit(X_train_candidates, y_train)

print(f'Features successfully binned: {len(binner.fitted_features_)}')
print(f'Features skipped: {len(candidates) - len(binner.fitted_features_)}')

Features successfully binned: 97
Features skipped: 0


## Step 5: IV Summary and Feature Selection

Information Value (IV) quantifies predictive power:
- IV < 0.02: Not predictive (drop)
- IV 0.02–0.10: Weak predictor (flag for review)
- IV 0.10–0.30: Medium predictor (include)
- IV 0.30–0.50: Strong predictor (include)
- IV > 0.50: Suspicious — investigate for data leakage

In [6]:
iv_df = binner.iv_summary()

print('IV Summary Table (all features):')
print('=' * 70)
for _, row in iv_df.iterrows():
    marker = ''
    if row['selection_status'] == 'suspicious_check_leakage':
        marker = ' *** CHECK LEAKAGE ***'
    elif row['selection_status'] == 'drop_not_predictive':
        marker = ' (drop)'
    elif row['selection_status'] == 'weak':
        marker = ' (weak)'
    print(f"  {row['feature']:45s} IV={row['iv']:.6f}  [{row['selection_status']}]{marker}")

print(f'\nIV Summary Statistics:')
print(f'  Mean IV:   {iv_df["iv"].mean():.4f}')
print(f'  Median IV: {iv_df["iv"].median():.4f}')
print(f'\nCount by selection status:')
for status, count in iv_df['selection_status'].value_counts().items():
    print(f'  {status}: {count}')

IV Summary Table (all features):
  term                                          IV=0.240284  [medium]
  loan_to_income                                IV=0.127054  [medium]
  fico_range_low                                IV=0.121740  [medium]
  fico_range_high                               IV=0.121740  [medium]
  fico_avg                                      IV=0.121740  [medium]
  installment_to_income                         IV=0.084641  [weak] (weak)
  acc_open_past_24mths                          IV=0.081087  [weak] (weak)
  dti                                           IV=0.075813  [weak] (weak)
  num_tl_op_past_12m                            IV=0.058084  [weak] (weak)
  bc_open_to_buy                                IV=0.055918  [weak] (weak)
  avg_cur_bal                                   IV=0.041534  [weak] (weak)
  total_bc_limit                                IV=0.039291  [weak] (weak)
  mo_sin_rcnt_tl                                IV=0.039144  [weak] (weak)
  tot_hi_cred_lim

## Step 6: Discuss High-IV Features and Scorecard Exclusions

### Why exclude `grade`, `sub_grade`, and `int_rate`?

These features will have **very high IV** (likely > 0.5) because they are
assigned by LendingClub's own credit model based on the borrower's risk profile.
Including them in a scorecard creates a **circular dependency**: the model would
essentially learn "high-risk grades default more" rather than learning from
underlying borrower characteristics.

- `grade`/`sub_grade`: Assigned by LC's credit algorithm
- `int_rate`: Directly determined by grade (A=low rate, G=high rate)

**Decision**: Exclude from scorecard (Notebook 03). Keep for ML models (Notebook 04)
where interpretability is less critical.

### Leakage Investigation
Any feature with IV > 0.5 that is NOT grade/int_rate should be investigated.
Common culprits: `total_pymnt`, `last_pymnt_amnt`, `last_fico_range_*`.

In [7]:
# Check which of the excluded features would have had high IV
# by running WOE binning on grade-related features separately
grade_feats = ['int_rate']
grade_binner = WOEBinner(max_bins=10, random_state=RANDOM_STATE)
grade_binner.fit(train[grade_feats], y_train)
grade_iv = grade_binner.iv_summary()

print('IV for excluded scorecard features (for documentation):')
for _, row in grade_iv.iterrows():
    print(f"  {row['feature']}: IV={row['iv']:.4f} — EXCLUDED (credit decision output)")

# Check for any suspicious features in the main selection
suspicious = iv_df[iv_df['selection_status'] == 'suspicious_check_leakage']
if len(suspicious) > 0:
    print(f'\nFeatures with IV > 0.50 (investigate for leakage):')
    for _, row in suspicious.iterrows():
        print(f"  {row['feature']}: IV={row['iv']:.4f}")
else:
    print('\nNo features with suspicious IV > 0.50 (good — no leakage detected).')

IV for excluded scorecard features (for documentation):
  int_rate: IV=0.4638 — EXCLUDED (credit decision output)

No features with suspicious IV > 0.50 (good — no leakage detected).


## Step 7: Final Feature Selection (IV-Based)

Select features with IV >= 0.02 (weak or stronger). Features with IV < 0.02
have negligible predictive power and are dropped.

In [8]:
# Select features with IV >= 0.02
selected = iv_df[iv_df['iv'] >= 0.02].copy()
dropped = iv_df[iv_df['iv'] < 0.02].copy()

selected_features = selected['feature'].tolist()

print(f'Features selected (IV >= 0.02): {len(selected_features)}')
print(f'Features dropped  (IV <  0.02): {len(dropped)}')

print(f'\nSelected features by IV tier:')
for status in ['strong', 'medium', 'weak', 'suspicious_check_leakage']:
    tier = selected[selected['selection_status'] == status]
    if len(tier) > 0:
        print(f'\n  {status.upper()} ({len(tier)}):')
        for _, row in tier.iterrows():
            print(f"    {row['feature']:40s} IV={row['iv']:.4f}")

if len(dropped) > 0:
    print(f'\nDropped features ({len(dropped)}):')
    for _, row in dropped.iterrows():
        print(f"    {row['feature']:40s} IV={row['iv']:.6f}")

Features selected (IV >= 0.02): 33
Features dropped  (IV <  0.02): 64

Selected features by IV tier:

  MEDIUM (5):
    term                                     IV=0.2403
    loan_to_income                           IV=0.1271
    fico_range_low                           IV=0.1217
    fico_range_high                          IV=0.1217
    fico_avg                                 IV=0.1217

  WEAK (28):
    installment_to_income                    IV=0.0846
    acc_open_past_24mths                     IV=0.0811
    dti                                      IV=0.0758
    num_tl_op_past_12m                       IV=0.0581
    bc_open_to_buy                           IV=0.0559
    avg_cur_bal                              IV=0.0415
    total_bc_limit                           IV=0.0393
    mo_sin_rcnt_tl                           IV=0.0391
    tot_hi_cred_lim                          IV=0.0372
    funded_amnt                              IV=0.0368
    loan_amnt                                

## Step 8: Validate Monotonicity of Bad Rates

For scorecard development, WOE bins should show monotonic bad rates.
Minor violations are acceptable and documented; major violations
require re-binning or feature transformation.

In [9]:
mono_results = []
for feat in selected_features:
    info = binner.check_monotonicity(feat)
    mono_results.append({
        'feature': feat,
        'monotonic': info['monotonic'],
        'direction': info['direction'],
        'n_bins': len(info['rates']),
    })

mono_df = pd.DataFrame(mono_results)
n_mono = mono_df['monotonic'].sum()
n_total = len(mono_df)

print(f'Monotonicity Check: {n_mono}/{n_total} features are monotonic ({n_mono/n_total:.0%})')
print()

violations = mono_df[~mono_df['monotonic']]
if len(violations) > 0:
    print(f'Non-monotonic features ({len(violations)}):')
    for _, row in violations.iterrows():
        print(f"  {row['feature']}: {row['direction']} ({row['n_bins']} bins)")
    print('\nNote: Minor monotonicity violations are acceptable in practice.')
    print('These features are still included — the WOE transformation handles the ordering.')
else:
    print('All selected features have monotonic bad rates across bins.')

Monotonicity Check: 33/33 features are monotonic (100%)

All selected features have monotonic bad rates across bins.


### Detailed Bin Tables — Top 10 Features by IV

For each of the top 10 features, display the full bin table showing:
bin edges, observation count, event/non-event counts, bad rate, WOE, and IV component.
This is the core artifact of scorecard development — each row represents a bin
that will map to scorecard points in Notebook 03.

In [10]:
# Display detailed bin tables for top 10 features
top10 = selected.head(10)['feature'].tolist()

for feat in top10:
    bt = binner.get_bin_table(feat)
    iv_val = binner.iv_values_[feat]
    mono = binner.check_monotonicity(feat)

    print(f'\n{"="*80}')
    print(f'{feat}  |  IV = {iv_val:.4f}  |  {mono["direction"].upper()}  |  Bins: {len(bt)}')
    print(f'{"="*80}')

    display_bt = bt[['bin_label', 'count', 'event_count', 'non_event_count',
                      'event_rate', 'woe', 'iv_component']].copy()
    display_bt['event_rate'] = display_bt['event_rate'].map('{:.2%}'.format)
    display_bt['woe'] = display_bt['woe'].map('{:+.4f}'.format)
    display_bt['iv_component'] = display_bt['iv_component'].map('{:.6f}'.format)
    display_bt.columns = ['Bin Range', 'Count', 'Events', 'Non-Events',
                           'Bad Rate', 'WOE', 'IV Component']
    print(display_bt.to_string(index=False))



term  |  IV = 0.2403  |  INCREASING  |  Bins: 2
    Bin Range  Count  Events  Non-Events Bad Rate     WOE IV Component
(-inf, 48.00] 618584   85955      532629   13.90% +0.3362     0.075824
 (48.00, inf) 208022   66349      141673   31.90% -0.7292     0.164460

loan_to_income  |  IV = 0.1271  |  INCREASING  |  Bins: 10
   Bin Range  Count  Events  Non-Events Bad Rate     WOE IV Component
(-inf, 0.07]  58528    6580       51948   11.24% +0.5784     0.019571
(0.07, 0.12] 110990   14115       96875   12.72% +0.4384     0.022353
(0.12, 0.16] 132791   18842      113949   14.19% +0.3119     0.014120
(0.16, 0.19]  82387   13052       69335   15.84% +0.1822     0.003121
(0.19, 0.22]  65905   11463       54442   17.39% +0.0702     0.000384
(0.22, 0.26] 106823   20290       86533   18.99% -0.0374     0.000183
(0.26, 0.30]  62638   13332       49306   21.28% -0.1799     0.002593
(0.30, 0.33]  67176   16135       51041   24.02% -0.3362     0.010167
(0.33, 0.40]  72891   19467       53424   26.71%

## Step 9: WOE Bin Plots (Top Features)

Visual inspection of WOE patterns and bad rates for the highest-IV features.

In [11]:
# Plot top 12 features by IV
top_features = selected.head(12)['feature'].tolist()

fig, axes = plt.subplots(4, 3, figsize=(20, 24))
for i, feat in enumerate(top_features):
    ax = axes.flatten()[i]
    binner.plot_bad_rate(feat, ax=ax)

plt.suptitle('Bad Rate by Bin — Top 12 Features by IV', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(DATA_RESULTS_PATH / 'woe_bad_rate_top12.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: WOE Transformation

Transform selected features to WOE values. The binner was fit on training data only —
now we apply it to train, validation, and test sets.

**Critical**: This transformation encodes each feature's relationship with the target
into a single numeric value per bin, enabling logistic regression to work with
optimally binned features.

In [12]:
# Select only the features that passed IV screening
# Create a new binner fitted only on selected features for clean transformation
X_train_sel = train[selected_features]
X_val_sel = val[selected_features]
X_test_sel = test[selected_features]

# Transform
train_woe = binner.transform(X_train_sel)
val_woe = binner.transform(X_val_sel)
test_woe = binner.transform(X_test_sel)

# Add target
train_woe[TARGET_COL] = y_train.values
val_woe[TARGET_COL] = y_val.values
test_woe[TARGET_COL] = y_test.values

# Add macro features (pass through without WOE transformation)
for feat in FRED_SERIES:
    if feat in train.columns:
        train_woe[feat] = train[feat].values
        val_woe[feat] = val[feat].values
        test_woe[feat] = test[feat].values

# Add binary flags (pass through without WOE transformation)
flag_cols = [c for c in train.columns
             if c.startswith(('has_', 'no_', 'emp_length_unknown', 'installment_features'))
             and c.endswith(('_missing', '_flag', '_unknown', '_delinq', '_history',
                             '_record', '_derog'))]
# Also include engineered binary flags
extra_flags = ['delinq_flag', 'recent_inquiry_flag', 'high_dti_flag']
flag_cols = list(set(flag_cols + extra_flags) & set(train.columns))

for feat in flag_cols:
    if feat not in train_woe.columns:
        train_woe[feat] = train[feat].values
        val_woe[feat] = val[feat].values
        test_woe[feat] = test[feat].values

print(f'Train WOE shape: {train_woe.shape}')
print(f'Val WOE shape:   {val_woe.shape}')
print(f'Test WOE shape:  {test_woe.shape}')
print(f'\nColumns: {list(train_woe.columns)}')
print(f'\nSample WOE values (first 5 rows):')
print(train_woe.head())

Train WOE shape: (826606, 49)
Val WOE shape:   (293105, 49)
Test WOE shape:  (225639, 49)

Columns: ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'installment', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'revol_util', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_actv_rev_tl', 'num_rev_tl_bal_gt_0', 'num_tl_op_past_12m', 'percent_bc_gt_75', 'tot_hi_cred_lim', 'total_bc_limit', 'fico_avg', 'loan_to_income', 'installment_to_income', 'high_dti_flag', 'default', 'UNRATE', 'CSUSHPINSA', 'A191RL1Q225SBEA', 'CPIAUCSL', 'DFF', 'UMCSENT', 'has_recent_bc_delinq', 'has_major_derog', 'recent_inquiry_flag', 'no_revol_delinq', 'no_delinq_history', 'installment_features_missing', 'delinq_flag', 'has_public_record', 'emp_length_unknown']

Sample WOE values (first 5 rows):


## Step 11: IV Summary Visualization

Horizontal bar chart showing IV for all selected features, color-coded by tier.

In [13]:
# IV bar chart
plot_df = selected.copy()
color_map = {
    'strong': '#2ca02c',
    'medium': '#1f77b4',
    'weak': '#ff7f0e',
    'suspicious_check_leakage': '#d62728',
    'drop_not_predictive': '#7f7f7f',
}
colors = [color_map.get(s, '#7f7f7f') for s in plot_df['selection_status']]

fig, ax = plt.subplots(figsize=(12, max(8, len(plot_df) * 0.35)))
bars = ax.barh(range(len(plot_df)), plot_df['iv'].values, color=colors)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['feature'].values, fontsize=9)
ax.set_xlabel('Information Value (IV)')
ax.set_title('Feature Selection by Information Value')
ax.axvline(x=0.02, color='gray', linestyle='--', alpha=0.5, label='Min threshold (0.02)')
ax.axvline(x=0.1, color='blue', linestyle='--', alpha=0.3, label='Medium (0.10)')
ax.axvline(x=0.3, color='green', linestyle='--', alpha=0.3, label='Strong (0.30)')
ax.legend(loc='lower right')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_RESULTS_PATH / 'iv_summary_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12: Correlation Among Selected WOE Features

Check for multicollinearity among WOE-transformed features.
Highly correlated pairs (|r| > 0.80) may need one member dropped.

In [14]:
woe_only = [c for c in train_woe.columns
            if c not in [TARGET_COL] + FRED_SERIES + flag_cols]
corr = train_woe[woe_only].corr()

# Extract high-correlation pairs
pairs = []
for i in range(len(woe_only)):
    for j in range(i+1, len(woe_only)):
        r = corr.iloc[i, j]
        if abs(r) > 0.70:
            pairs.append((woe_only[i], woe_only[j], round(r, 3)))

if pairs:
    print('Highly correlated WOE feature pairs (|r| > 0.70):')
    for f1, f2, r in sorted(pairs, key=lambda x: -abs(x[2])):
        print(f'  {f1:35s} ↔ {f2:35s}  r={r:+.3f}')
else:
    print('No highly correlated WOE feature pairs found (|r| > 0.70).')

# Heatmap
if len(woe_only) > 0:
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                xticklabels=True, yticklabels=True, ax=ax, fmt='.1f',
                annot=len(woe_only) <= 20, square=True)
    ax.set_title('Correlation Matrix — WOE-Transformed Features')
    plt.tight_layout()
    plt.savefig(DATA_RESULTS_PATH / 'woe_correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

Highly correlated WOE feature pairs (|r| > 0.70):
  fico_range_low                      ↔ fico_range_high                      r=+1.000
  fico_range_low                      ↔ fico_avg                             r=+1.000
  fico_range_high                     ↔ fico_avg                             r=+1.000
  loan_amnt                           ↔ funded_amnt                          r=+0.999
  funded_amnt                         ↔ funded_amnt_inv                      r=+0.994
  loan_amnt                           ↔ funded_amnt_inv                      r=+0.993
  num_actv_rev_tl                     ↔ num_rev_tl_bal_gt_0                  r=+0.980
  tot_cur_bal                         ↔ tot_hi_cred_lim                      r=+0.957
  loan_to_income                      ↔ installment_to_income                r=+0.928
  tot_cur_bal                         ↔ avg_cur_bal                          r=+0.860
  avg_cur_bal                         ↔ tot_hi_cred_lim                      r=+0.839
  to

## Step 13: Save All Outputs

Save fitted binner, IV summary, and WOE-transformed datasets.

In [15]:
# Save WOE binner object
with open(DATA_PROCESSED_PATH / 'woe_binning_results.pkl', 'wb') as f:
    pickle.dump(binner, f)
print(f'Saved: woe_binning_results.pkl')

# Save IV summary
iv_df.to_csv(DATA_PROCESSED_PATH / 'iv_summary.csv', index=False)
print(f'Saved: iv_summary.csv ({len(iv_df)} features)')

# Save WOE-transformed datasets
train_woe.to_parquet(DATA_PROCESSED_PATH / 'train_woe.parquet', index=False)
val_woe.to_parquet(DATA_PROCESSED_PATH / 'val_woe.parquet', index=False)
test_woe.to_parquet(DATA_PROCESSED_PATH / 'test_woe.parquet', index=False)
print(f'Saved: train_woe.parquet ({train_woe.shape})')
print(f'Saved: val_woe.parquet ({val_woe.shape})')
print(f'Saved: test_woe.parquet ({test_woe.shape})')

print(f'\n{"="*60}')
print(f'SESSION 2 COMPLETE')
print(f'{"="*60}')
print(f'Features fitted:  {len(binner.fitted_features_)}')
print(f'Features selected (IV >= 0.02): {len(selected_features)}')
print(f'WOE columns in output: {len(woe_only)}')
print(f'Macro features carried through: {[s for s in FRED_SERIES if s in train_woe.columns]}')
print(f'Binary flags carried through: {len(flag_cols)}')

Saved: woe_binning_results.pkl
Saved: iv_summary.csv (97 features)


Saved: train_woe.parquet ((826606, 49))
Saved: val_woe.parquet ((293105, 49))
Saved: test_woe.parquet ((225639, 49))

SESSION 2 COMPLETE
Features fitted:  97
Features selected (IV >= 0.02): 33
WOE columns in output: 32
Macro features carried through: ['UNRATE', 'CSUSHPINSA', 'A191RL1Q225SBEA', 'CPIAUCSL', 'DFF', 'UMCSENT']
Binary flags carried through: 10
